# fair value estimator comparison

**goal:** decide whether the current fair value (fv) estimators used in trader_v6 are the best choice for five downstream tasks:
1. regime detection (calm vs active vs volatile)
2. inventory management and quote skewing
3. timing trade entries and exits
4. spread size detection
5. general price prediction accuracy

**what we are comparing:**

| estimator | short name | description |
|---|---|---|
| exponential moving average (span=7) | ema-7 | current trader_v6 fv for tomatoes |
| hard-coded constant 10000 | constant | current trader_v6 fv for emeralds |
| kalman filter | kalman | adaptive bayesian tracker |
| micro price | micro | volume-weighted best bid/ask |
| wall mid | wall-mid | size-weighted average across up to 3 book levels |
| ornstein-uhlenbeck fit | ou | parametric mean-reversion model |
| ema-7 / ema-20 crossover trend follower | hybrid | blends short and long ema signals |

**three metrics used to rank the estimators:**
1. **mean reversion half-life** -- how quickly does the error between estimator and actual price decay back to zero? a shorter half-life means the estimator tracks the price faster, which is good for inventory management.
2. **z-score normality of errors** -- are the prediction errors well-behaved and gaussian? non-gaussian errors suggest fat tails or systematic bias that could cause underpriced risk.
3. **directional prediction accuracy** -- when the estimator says price > fv, what is the probability that the next 5 ticks are net negative (i.e., price reverts)? higher probability = better signal quality for timing entries.

---

**data sources used:**
- `prices_round_0_day_-1.csv` and `prices_round_0_day_-2.csv`: full order book snapshots (bid/ask levels 1-3, mid price) across ~20,000 ticks per product per day
- `Json_Log_6.json`: trading log from the v6 bot, containing the same market data already merged into a single activities log

we use the two-day csv files for maximum data volume (20,000 ticks per product) and treat the json log as a cross-check.

## section 1: imports and setup

all analysis is done with standard scientific python libraries. no external trading libraries are required.

- `pandas` and `numpy` for data manipulation
- `scipy.stats` for statistical tests (shapiro-wilk, adf, linear regression)
- `matplotlib` for all plots
- `statsmodels` for the augmented dickey-fuller (adf) stationarity test used in half-life estimation

In [ ]:
import json
import io
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# statsmodels is used only for the ADF test and OLS regression in half-life estimation
try:
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.regression.linear_model import OLS
    from statsmodels.tools import add_constant
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("warning: statsmodels not found. half-life will use a manual OLS fallback.")

# plot style: clean, minimal, readable
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

print("imports complete.")

## section 2: load and prepare data

we load all four csv files and concatenate them into two long time series: one for tomatoes and one for emeralds.

**important note on timestamp continuity:**
the two days (day -2 and day -1) are separate trading sessions. timestamps reset to 0 at the start of each day. we sort within each day by timestamp, then concatenate, so the resulting series is ordered but not literally continuous in wall-clock time. this matters when computing lagged differences -- we must never compute a diff across the day boundary. we handle this by keeping a `day` column as a grouping key.

**wall mid calculation:**
the dataset provides up to 3 bid/ask price levels with associated volumes. the wall mid weights each level by its size:

wall_mid = sum(ask_price_i * bid_vol_i + bid_price_i * ask_vol_i for i in levels) / sum(bid_vol_i + ask_vol_i for i in levels)

intuitively this says: levels with more volume on the opposite side have more pull on the fair price.

**micro price calculation:**
the micro price weights only the best bid and ask by the volume on the *opposite* side:

micro = (ask_price_1 * bid_vol_1 + bid_price_1 * ask_vol_1) / (bid_vol_1 + ask_vol_1)

when the bid volume is large relative to the ask volume, the micro price sits closer to the ask (the market is leaning bullish). this is the single most commonly used order-book-derived fair value estimate in high-frequency trading literature.

In [ ]:
# ----------------------------------------------------------------
# load raw csv files
# ----------------------------------------------------------------

# update these paths to match wherever you saved the csv files
PATH_P1 = 'prices_round_0_day_-1.csv'
PATH_P2 = 'prices_round_0_day_-2.csv'

p1 = pd.read_csv(PATH_P1, sep=';')
p2 = pd.read_csv(PATH_P2, sep=';')
prices_raw = pd.concat([p2, p1], ignore_index=True)

# sort within each day to ensure time order is correct
prices_raw = prices_raw.sort_values(['day', 'timestamp']).reset_index(drop=True)

print(f"total rows loaded: {len(prices_raw)}")
print(f"products: {prices_raw['product'].unique()}")
print(f"days: {sorted(prices_raw['day'].unique())}")
prices_raw.head(3)

In [ ]:
# ----------------------------------------------------------------
# compute micro price and wall mid for each row
# ----------------------------------------------------------------

df = prices_raw.copy()

# micro price: uses only level 1 bid and ask with their volumes
# formula: weight best ask by bid volume and best bid by ask volume
# this shifts the midpoint toward whichever side has thinner supply
df['micro_price'] = (
    df['ask_price_1'] * df['bid_volume_1'] +
    df['bid_price_1'] * df['ask_volume_1']
) / (df['bid_volume_1'] + df['ask_volume_1'])

# wall mid: extends micro price logic to all available book levels
# level 3 data is sparse (~3.6% of rows) so we use it only when available

def compute_wall_mid(row):
    """
    compute a volume-weighted fair value from up to 3 book levels.
    each ask level is weighted by the bid volume at that level and vice versa.
    levels with no data (NaN) are skipped.
    """
    numerator = 0.0
    denominator = 0.0
    for i in [1, 2, 3]:
        bp = row.get(f'bid_price_{i}', np.nan)
        bv = row.get(f'bid_volume_{i}', np.nan)
        ap = row.get(f'ask_price_{i}', np.nan)
        av = row.get(f'ask_volume_{i}', np.nan)
        if pd.notna(bp) and pd.notna(bv) and pd.notna(ap) and pd.notna(av):
            numerator += ap * bv + bp * av
            denominator += bv + av
    if denominator == 0:
        return np.nan
    return numerator / denominator

df['wall_mid'] = df.apply(compute_wall_mid, axis=1)

# spread: best ask minus best bid (tick-level spread)
df['spread'] = df['ask_price_1'] - df['bid_price_1']

print("feature columns added: micro_price, wall_mid, spread")
print(f"wall_mid non-null: {df['wall_mid'].notna().sum()} of {len(df)} rows")
df[['product','timestamp','mid_price','micro_price','wall_mid','spread']].head(6)

## section 3: compute all fair value estimators

we compute each estimator as a time series aligned tick-by-tick with the observed mid price. all estimators are computed separately for each product and each day (so no estimator leaks information across the day boundary).

### ema-7
the exponential moving average with span 7 gives decay weight alpha = 2/(7+1) = 0.25 per tick. this means the most recent tick gets 25% weight and older ticks decay geometrically. with span 7, the effective memory is roughly 7 ticks. this is the current estimator used in trader_v6 for tomatoes.

### constant (10000)
the emerald fair value is hardcoded at 10000. this is appropriate only if the market is known to mean-revert to a fixed level. we will test whether this assumption holds in the data.

### kalman filter
the kalman filter maintains a probabilistic belief about the true price, represented as a gaussian with mean (the estimate) and variance (our uncertainty). at each tick it does two steps:
- **predict**: carry forward last estimate, inflate variance by process noise q
- **update**: blend prediction with new observation weighted by signal-to-noise ratio

the key hyperparameter is the ratio q/r where q is process noise and r is observation noise. a higher q/r makes the filter more responsive (like a lower ema span). we choose q=0.5, r=1.0 as a baseline -- these can be tuned.

**limitation:** the kalman filter here assumes a random walk (no drift). a more complete implementation would also model the velocity (first difference) as a state variable, but this requires more data to converge.

### micro price
already computed in section 2. this is purely derived from the order book at each tick with no smoothing.

### wall mid
already computed in section 2. similar to micro price but uses deeper book levels.

### ornstein-uhlenbeck (ou) rolling fit
the ou process is the continuous-time model for mean reversion:

dx = theta * (mu - x) * dt + sigma * dW

where theta is the speed of reversion, mu is the long-run mean, and sigma is volatility. we fit this to a rolling window of 50 ticks using maximum likelihood (which simplifies to an OLS regression of price changes on price levels). the fitted mu becomes the fair value estimate.

**limitation:** with only 50-tick windows, the ou fit can be noisy. the half-life 1/theta must be interpreted carefully -- values near or above the window length are unreliable.

### hybrid ema crossover
the hybrid combines a short ema (span 7) and a long ema (span 20). when the short ema is above the long ema, we say the price is in an uptrend, so we shade the fair value estimate upward by a fraction of the crossover gap. conversely for downtrends.

fv_hybrid = ema_short + alpha * (ema_short - ema_long)

with alpha = 0.5. this makes the estimator trend-following rather than mean-reverting, which may be useful for timing entries but harmful for inventory management.

In [ ]:
# ----------------------------------------------------------------
# helper: kalman filter (scalar, random walk model)
# ----------------------------------------------------------------

def kalman_filter(prices, q=0.5, r=1.0):
    """
    simple 1-d kalman filter for a scalar price series.

    parameters
    ----------
    prices : array-like of observed prices
    q      : process noise variance (how much the true price can move per tick)
    r      : observation noise variance (how noisy our price measurement is)

    returns
    -------
    array of filtered (estimated true) prices, same length as input

    intuition
    ---------
    the filter balances two sources of information:
      - what we predicted the price would be (based on last estimate)
      - what we actually observed
    the kalman gain k = p / (p + r) tells us how much to trust the new observation.
    if r is small (observations are precise), k is close to 1 and we follow observations tightly.
    if q is small (price barely moves), k stays small and we trust our prior more.
    """
    n = len(prices)
    fv = np.zeros(n)
    p = r  # initial uncertainty: assume same magnitude as observation noise
    x = prices[0]  # initialise to first observation

    for i, z in enumerate(prices):
        # predict step: propagate variance forward (no drift in this model)
        p = p + q
        # update step: compute kalman gain and update estimate
        k = p / (p + r)
        x = x + k * (z - x)
        p = (1 - k) * p
        fv[i] = x

    return fv


# ----------------------------------------------------------------
# helper: ornstein-uhlenbeck rolling fit
# ----------------------------------------------------------------

def ou_rolling_mu(prices, window=50):
    """
    fit an ou process to a rolling window and extract the long-run mean mu.

    the ou model in discrete time is:
        delta_x[t] = theta * (mu - x[t-1]) * dt + noise

    rearranged for OLS:
        delta_x[t] = a + b * x[t-1]  where a = theta*mu*dt, b = -theta*dt

    so mu = -a / b  (assuming b < 0, i.e., mean-reverting)

    if the window is non-mean-reverting (b >= 0), we fall back to the simple
    rolling mean of x over that window instead.

    limitation: this is a rough fit. the window of 50 ticks is short relative
    to slower reversion cycles, so mu estimates may lag. treat them as
    directional signals rather than precise fair values.
    """
    n = len(prices)
    mu_est = np.full(n, np.nan)

    for i in range(window, n):
        x = prices[i - window:i]
        dx = np.diff(x)
        x_lag = x[:-1]
        # OLS: dx = a + b * x_lag
        A = np.column_stack([np.ones(len(x_lag)), x_lag])
        try:
            coeffs, _, _, _ = np.linalg.lstsq(A, dx, rcond=None)
            a, b = coeffs
            if b < 0:  # mean-reverting regime
                mu_est[i] = -a / b
            else:  # not mean-reverting: fall back to rolling mean
                mu_est[i] = x.mean()
        except Exception:
            mu_est[i] = x.mean()

    return mu_est


print("estimator helper functions defined.")

In [ ]:
# ----------------------------------------------------------------
# compute all estimators for tomatoes (across both days)
# ----------------------------------------------------------------

results = {}  # will hold one dataframe per product

for product in ['TOMATOES', 'EMERALDS']:

    frames = []
    for day in sorted(df['day'].unique()):
        sub = df[(df['product'] == product) & (df['day'] == day)].copy().reset_index(drop=True)
        if len(sub) == 0:
            continue

        mid = sub['mid_price'].values

        # ema-7 (span=7, alpha=0.25, matches trader_v6 exactly)
        sub['fv_ema7'] = sub['mid_price'].ewm(span=7, adjust=False).mean()

        # constant fv (10000 for emeralds, use mean of first 10 ticks for tomatoes as baseline)
        if product == 'EMERALDS':
            sub['fv_const'] = 10000.0
        else:
            # for tomatoes we use the grand mean as the 'constant' baseline
            # this is not trader_v6 behaviour but gives a meaningful comparison
            sub['fv_const'] = mid.mean()

        # kalman filter (q=0.5, r=1.0)
        sub['fv_kalman'] = kalman_filter(mid, q=0.5, r=1.0)

        # micro price (already computed)
        sub['fv_micro'] = sub['micro_price']

        # wall mid (already computed)
        sub['fv_wallmid'] = sub['wall_mid']

        # ou rolling mean (window=50 ticks)
        sub['fv_ou'] = ou_rolling_mu(mid, window=50)

        # hybrid: ema7 + 0.5 * (ema7 - ema20) to tilt in the direction of the trend
        ema7  = sub['mid_price'].ewm(span=7,  adjust=False).mean()
        ema20 = sub['mid_price'].ewm(span=20, adjust=False).mean()
        sub['fv_hybrid'] = ema7 + 0.5 * (ema7 - ema20)

        frames.append(sub)

    results[product] = pd.concat(frames, ignore_index=True)
    print(f"{product}: {len(results[product])} rows, estimators computed.")

tom = results['TOMATOES']
em  = results['EMERALDS']

## section 4: metric 1 -- mean reversion half-life

### what this measures
the half-life of mean reversion tells us: if the price is currently N units above the fair value estimate, how many ticks does it take, on average, for half that gap to close?

a shorter half-life is better for a market maker because:
- it means the fv estimate is tracking the true equilibrium tightly
- inventory positions built up during brief deviations from fv will unwind faster
- the skew mechanism in trader_v6 can be calibrated more aggressively

### how we estimate it
we use the ornstein-uhlenbeck regression approach (sometimes called the half-life from the ar(1) coefficient):

1. compute the error series: e[t] = mid_price[t] - fv_estimate[t]
2. regress delta_e[t] on e[t-1]: delta_e = a + b * e[t-1]
3. the ou mean-reversion speed is theta = -b (per tick)
4. the half-life is log(2) / theta (in ticks)

if b >= 0, the error is not mean-reverting, which means the estimator is drifting away from the price rather than tracking it.

### limitations
- this is a linear approximation. real markets have nonlinear reversion dynamics, especially near position limits.
- the adf test we run alongside checks statistical significance. if the adf p-value > 0.05, the error series may be a unit root (non-stationary), and the half-life estimate is not reliable.
- book-anchored estimators (micro, wall-mid) have very small errors almost by definition since they are derived from the same book that produces the mid price. their short half-lives are partly an artefact.

In [ ]:
# ----------------------------------------------------------------
# half-life estimation via ou regression
# ----------------------------------------------------------------

ESTIMATORS = {
    'ema-7':    'fv_ema7',
    'constant': 'fv_const',
    'kalman':   'fv_kalman',
    'micro':    'fv_micro',
    'wall-mid': 'fv_wallmid',
    'ou':       'fv_ou',
    'hybrid':   'fv_hybrid',
}


def estimate_half_life(errors):
    """
    given a series of errors e[t] = actual_price - fv_estimate,
    fit the ou regression delta_e = a + b * e[t-1] and return:
      - half_life in ticks (log(2) / -b)
      - b coefficient (negative = mean reverting)
      - adf p-value (low = stationary = good)
    """
    e = errors.dropna().values
    if len(e) < 30:
        return np.nan, np.nan, np.nan

    delta_e = np.diff(e)
    e_lag   = e[:-1]

    # OLS: delta_e = a + b * e_lag
    A = np.column_stack([np.ones(len(e_lag)), e_lag])
    try:
        coeffs, _, _, _ = np.linalg.lstsq(A, delta_e, rcond=None)
    except Exception:
        return np.nan, np.nan, np.nan

    a, b = coeffs

    if b >= 0:
        # positive b means the error is explosive, not mean-reverting
        half_life = np.inf
    else:
        half_life = np.log(2) / (-b)

    # adf test: null hypothesis is unit root (non-stationary)
    # a low p-value (< 0.05) means we can reject the unit root and
    # confidently say the error series is stationary / mean-reverting
    adf_pval = np.nan
    if HAS_STATSMODELS and len(e) >= 20:
        try:
            adf_result = adfuller(e, maxlag=5, autolag='AIC')
            adf_pval = adf_result[1]
        except Exception:
            pass

    return half_life, b, adf_pval


# run for both products
hl_rows = []
for product, data_df in [('TOMATOES', tom), ('EMERALDS', em)]:
    for name, col in ESTIMATORS.items():
        if col not in data_df.columns:
            continue
        errors = data_df['mid_price'] - data_df[col]
        hl, b, pval = estimate_half_life(errors)
        hl_rows.append({
            'product':   product,
            'estimator': name,
            'half_life_ticks': round(hl, 2) if np.isfinite(hl) else 'inf',
            'b_coeff':   round(b, 5) if b is not None and not np.isnan(b) else np.nan,
            'adf_pval':  round(pval, 4) if pval is not None and not np.isnan(pval) else 'n/a',
        })

hl_df = pd.DataFrame(hl_rows)
print("\nhalf-life results:")
print(hl_df.to_string(index=False))

In [ ]:
# ----------------------------------------------------------------
# plot: half-life bar chart
# ----------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, product in zip(axes, ['TOMATOES', 'EMERALDS']):
    sub = hl_df[hl_df['product'] == product].copy()
    # replace inf with a large display value
    sub['hl_display'] = pd.to_numeric(sub['half_life_ticks'], errors='coerce').fillna(999)
    sub = sub.sort_values('hl_display')

    colors = ['#d62728' if e == 'ema-7' else
              '#2ca02c' if e == 'constant' else
              '#1f77b4' for e in sub['estimator']]

    bars = ax.bar(sub['estimator'], sub['hl_display'].clip(upper=500), color=colors, width=0.6)

    # annotate exact values
    for bar, val in zip(bars, sub['half_life_ticks']):
        label = str(val) if val != 'inf' else 'inf'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                label, ha='center', va='bottom', fontsize=8)

    ax.set_title(f'{product.lower()} -- half-life of fv error (ticks)\n'
                 'shorter = estimator tracks price faster')
    ax.set_ylabel('half-life (ticks)')
    ax.set_xlabel('estimator')
    ax.tick_params(axis='x', rotation=30)

    # legend: current trader_v6 estimator is highlighted in red/green
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d62728', label='current (ema-7)'),
        Patch(facecolor='#2ca02c', label='current (constant)'),
        Patch(facecolor='#1f77b4', label='alternative'),
    ]
    ax.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plt.savefig('halflife_comparison.png', bbox_inches='tight')
plt.show()
print("\ninterpretation:")
print("  book-derived estimators (micro, wall-mid) will always show very short half-lives")
print("  because they are computed from the same book that generates the mid price.")
print("  for inventory management, the more relevant comparison is among the smoother")
print("  estimators: ema-7, kalman, ou, and hybrid.")

## section 5: metric 2 -- z-score normality of errors

### what this measures
the z-score of each error is computed as:

z[t] = (mid_price[t] - fv[t]) / rolling_std(mid_price, 20)

if these z-scores are normally distributed (mean~0, std~1), the estimator is well-calibrated and the risk model can use a gaussian assumption safely.

if the z-scores have fat tails (kurtosis > 3) or are skewed (skewness != 0), the estimator is systematically missing something:
- right skew means the estimator tends to underestimate the price
- fat tails mean large errors occur more often than a gaussian would predict

### tests used
- **shapiro-wilk test**: tests whether the distribution is normal. p < 0.05 means we reject normality.
- **skewness and kurtosis**: describe the shape of the distribution directly.
- **qq plot**: visually compares the empirical quantiles to the theoretical normal quantiles. points on the diagonal line = normal.

### why this matters for trader_v6
trader_v6 uses the fv estimator to skew quotes when inventory builds up. if the errors are fat-tailed, the bot may be caught offside more often than expected during large price moves -- the skew mechanism will not trigger fast enough.

In [ ]:
# ----------------------------------------------------------------
# z-score normality analysis
# ----------------------------------------------------------------

Z_ROLLING_WINDOW = 20  # same as TOM_VOL_WINDOW in trader_v6

normality_rows = []

for product, data_df in [('TOMATOES', tom), ('EMERALDS', em)]:
    # rolling standard deviation of the mid price
    rolling_std = data_df['mid_price'].rolling(Z_ROLLING_WINDOW).std()

    for name, col in ESTIMATORS.items():
        if col not in data_df.columns:
            continue

        errors = data_df['mid_price'] - data_df[col]
        # avoid division by zero: only where rolling std is positive
        z = (errors / rolling_std).replace([np.inf, -np.inf], np.nan).dropna()

        if len(z) < 20:
            continue

        skew_val  = float(stats.skew(z))
        kurt_val  = float(stats.kurtosis(z, fisher=True))  # excess kurtosis (normal=0)
        sw_stat, sw_pval = stats.shapiro(z[:5000])  # shapiro limited to 5000 samples

        normality_rows.append({
            'product':   product,
            'estimator': name,
            'n':         len(z),
            'mean_z':    round(z.mean(), 4),
            'std_z':     round(z.std(), 4),
            'skewness':  round(skew_val, 3),
            'ex_kurtosis': round(kurt_val, 3),
            'shapiro_p': round(sw_pval, 4),
            'normal?':   'yes' if sw_pval > 0.05 else 'no',
        })

norm_df = pd.DataFrame(normality_rows)
print("z-score normality results:")
print(norm_df.to_string(index=False))
print()
print("key columns:")
print("  skewness:    0 is ideal. positive = right tail (underestimates price).")
print("  ex_kurtosis: 0 is ideal (normal). positive = fat tails.")
print("  shapiro_p:   > 0.05 means we cannot reject normality (good).")

In [ ]:
# ----------------------------------------------------------------
# plot: qq plots and histograms for tomatoes
# ----------------------------------------------------------------

# we show qq plots for all estimators for tomatoes (the more complex product)
# a perfect normal estimator produces points on the diagonal reference line

product = 'TOMATOES'
data_df = tom
rolling_std = data_df['mid_price'].rolling(Z_ROLLING_WINDOW).std()

est_names = list(ESTIMATORS.keys())
n_est = len(est_names)
fig, axes = plt.subplots(2, n_est, figsize=(18, 7))

for col_idx, (name, col) in enumerate(ESTIMATORS.items()):
    if col not in data_df.columns:
        continue
    errors = data_df['mid_price'] - data_df[col]
    z = (errors / rolling_std).replace([np.inf, -np.inf], np.nan).dropna().values

    # top row: histogram with normal overlay
    ax_hist = axes[0, col_idx]
    ax_hist.hist(z, bins=60, density=True, color='#4878cf', alpha=0.7, label='empirical')
    x_range = np.linspace(z.min(), z.max(), 200)
    ax_hist.plot(x_range, stats.norm.pdf(x_range, 0, 1), 'r-', lw=2, label='normal(0,1)')
    ax_hist.set_title(f'{name}\nerror dist', fontsize=9)
    ax_hist.set_xlabel('z-score', fontsize=8)
    if col_idx == 0:
        ax_hist.set_ylabel('density', fontsize=8)
        ax_hist.legend(fontsize=7)

    # bottom row: qq plot
    ax_qq = axes[1, col_idx]
    stats.probplot(z, dist='norm', plot=ax_qq)
    ax_qq.get_lines()[0].set(markersize=2, alpha=0.4, color='#4878cf')
    ax_qq.get_lines()[1].set(color='red', lw=1.5)
    ax_qq.set_title(f'{name}\nqq plot', fontsize=9)
    ax_qq.set_xlabel('theoretical quantiles', fontsize=8)
    if col_idx == 0:
        ax_qq.set_ylabel('sample quantiles', fontsize=8)

fig.suptitle('tomatoes: z-score error distributions (top: histogram vs normal, bottom: qq plot)\n'
             'ideal estimator: histogram matches red curve, qq points lie on diagonal',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('zscore_normality.png', bbox_inches='tight')
plt.show()

## section 6: metric 3 -- directional prediction accuracy

### what this measures
we ask a simple binary question: when the current price is above the fair value estimate (price > fv), what fraction of the time does the price decrease over the next 5 ticks?

formally:

accuracy = p(mid_price[t+5] < mid_price[t]  |  mid_price[t] > fv[t])

a value near 0.5 means the estimator has no predictive power. a value near 0.7 or higher means the estimator is a useful signal for timing exits (when long) or entries (when short).

we also compute the symmetric case: when price < fv, what fraction of the time does price increase over the next 5 ticks?

### why 5 ticks?
5 ticks is approximately 500ms given the 100ms tick spacing in this dataset. this is a natural horizon for a market-maker who wants to decide whether to lean into or away from a position. longer horizons tend to have lower accuracy because price mean reversion competes with trend.

### limitations
- this is an in-sample test. the accuracy figures will be optimistic because the estimators were fit on the same data.
- when signal conditions are rare (few ticks where price > fv), the accuracy estimate has high variance. we report the sample size alongside each estimate.
- the 5-tick horizon is arbitrary. we recommend testing at 1, 3, 5, and 10 ticks to check robustness.

In [ ]:
# ----------------------------------------------------------------
# directional prediction accuracy
# ----------------------------------------------------------------

HORIZONS = [1, 3, 5, 10]  # ticks ahead to check

accuracy_rows = []

for product, data_df in [('TOMATOES', tom), ('EMERALDS', em)]:
    mid = data_df['mid_price'].values

    for name, col in ESTIMATORS.items():
        if col not in data_df.columns:
            continue
        fv = data_df[col].values

        for h in HORIZONS:
            n = len(mid) - h  # we can only look ahead h ticks from here
            if n < 50:
                continue

            # signal: price above fv right now
            above_fv = mid[:n] > fv[:n]
            # outcome: price falls over next h ticks
            price_fell = mid[h:n+h] < mid[:n]

            # signal: price below fv right now
            below_fv = mid[:n] < fv[:n]
            # outcome: price rose over next h ticks
            price_rose = mid[h:n+h] > mid[:n]

            n_above = above_fv.sum()
            n_below = below_fv.sum()

            acc_above = price_fell[above_fv].mean() if n_above > 0 else np.nan
            acc_below = price_rose[below_fv].mean() if n_below > 0 else np.nan
            # combined accuracy: pooling both signals
            acc_combined = np.nan
            if n_above + n_below > 0:
                correct = price_fell[above_fv].sum() + price_rose[below_fv].sum()
                total   = n_above + n_below
                acc_combined = correct / total

            accuracy_rows.append({
                'product':    product,
                'estimator':  name,
                'horizon':    h,
                'n_above':    int(n_above),
                'acc_above':  round(acc_above, 3) if not np.isnan(acc_above) else np.nan,
                'n_below':    int(n_below),
                'acc_below':  round(acc_below, 3) if not np.isnan(acc_below) else np.nan,
                'acc_combined': round(acc_combined, 3) if not np.isnan(acc_combined) else np.nan,
            })

acc_df = pd.DataFrame(accuracy_rows)

# show h=5 results (our primary horizon)
print("directional accuracy at horizon = 5 ticks:")
print(acc_df[acc_df['horizon'] == 5].drop(columns=['horizon']).to_string(index=False))
print()
print("note: a random baseline would score 0.5. values above 0.55 are potentially meaningful.")

In [ ]:
# ----------------------------------------------------------------
# plot: accuracy vs horizon for tomatoes
# ----------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))

colors_map = {
    'ema-7':    '#d62728',
    'constant': '#2ca02c',
    'kalman':   '#1f77b4',
    'micro':    '#ff7f0e',
    'wall-mid': '#9467bd',
    'ou':       '#8c564b',
    'hybrid':   '#e377c2',
}

sub = acc_df[(acc_df['product'] == 'TOMATOES')]
for name in ESTIMATORS.keys():
    est_sub = sub[sub['estimator'] == name].sort_values('horizon')
    ax.plot(est_sub['horizon'], est_sub['acc_combined'],
            marker='o', linewidth=2, color=colors_map.get(name, 'gray'),
            label=name)

ax.axhline(0.5, color='black', linestyle='--', linewidth=1.2, label='random baseline (0.5)')
ax.set_xlabel('prediction horizon (ticks)')
ax.set_ylabel('combined directional accuracy')
ax.set_title('tomatoes: directional accuracy vs prediction horizon\n'
             'higher = estimator better predicts direction of next move')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0.35, 0.75)
ax.set_xticks(HORIZONS)
plt.tight_layout()
plt.savefig('accuracy_vs_horizon.png', bbox_inches='tight')
plt.show()

## section 7: visual comparison of estimators on raw price

before drawing conclusions, it helps to look at what each estimator actually produces on a sample of the price series. this section plots a 500-tick window of tomato prices alongside all fv estimates.

**what to look for:**
- estimators that track price closely (micro, wall-mid) will appear almost indistinguishable from the mid price
- the ema-7 will be a slightly smoother version of the mid price
- the kalman filter will adapt more slowly in volatile periods
- the ou estimator may behave erratically in the first 50 ticks while it warms up
- the hybrid will trend in the direction of momentum, visibly deviating from the price during reversals

In [ ]:
# ----------------------------------------------------------------
# time series visual: first 500 ticks of tomatoes (day -2)
# ----------------------------------------------------------------

sample = tom[tom['day'] == tom['day'].min()].head(500).copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# top panel: micro, wall-mid, mid price (book-derived estimators)
axes[0].plot(sample.index, sample['mid_price'], 'k-', lw=1.5, label='mid price', zorder=5)
axes[0].plot(sample.index, sample['fv_micro'],   color='#ff7f0e', lw=1, alpha=0.8, label='micro price')
axes[0].plot(sample.index, sample['fv_wallmid'], color='#9467bd', lw=1, alpha=0.8, label='wall mid')
axes[0].set_ylabel('price')
axes[0].set_title('book-derived estimators (micro, wall-mid) vs mid price')
axes[0].legend(fontsize=9)

# middle panel: smoothed estimators
axes[1].plot(sample.index, sample['mid_price'], 'k-', lw=1.5, label='mid price', zorder=5)
axes[1].plot(sample.index, sample['fv_ema7'],   color='#d62728', lw=1.5, label='ema-7 (current)')
axes[1].plot(sample.index, sample['fv_kalman'], color='#1f77b4', lw=1.5, label='kalman')
axes[1].plot(sample.index, sample['fv_hybrid'], color='#e377c2', lw=1.5, linestyle='--', label='hybrid')
axes[1].set_ylabel('price')
axes[1].set_title('smoothed estimators: ema-7, kalman, hybrid')
axes[1].legend(fontsize=9)

# bottom panel: ou mean estimate (longer warm-up)
axes[2].plot(sample.index, sample['mid_price'], 'k-', lw=1.5, label='mid price', zorder=5)
axes[2].plot(sample.index, sample['fv_ou'], color='#8c564b', lw=1.5, label='ou rolling mean')
axes[2].set_ylabel('price')
axes[2].set_xlabel('tick index (within sample)')
axes[2].set_title('ou rolling mean (50-tick window, warm-up visible at start)')
axes[2].legend(fontsize=9)

fig.suptitle('tomatoes: fair value estimators on first 500 ticks (day -2)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('timeseries_estimators.png', bbox_inches='tight')
plt.show()

## section 8: regime detection capability

trader_v6 uses the error between fv and mid price to detect three regimes: calm, active, volatile. this section tests how well each estimator separates these regimes.

we define regimes using the rolling 20-tick standard deviation of mid price returns (same as TOM_VOL_WINDOW in trader_v6):
- calm: rolling std < 1.5 (matching TOM_VOL_CALM_THRESH)
- active: 1.5 <= rolling std < 2.5 (matching TOM_VOL_ACTIVE_THRESH)
- volatile: rolling std >= 2.5

for each estimator we compute the mean absolute error (mae) separately within each regime. an estimator is better for regime detection if:
1. its mae is low in calm regimes (tracks the price when it's quiet)
2. its mae increases in volatile regimes (so it doesn't blindly chase the price during noise spikes)

**the ideal fv for regime detection** is one that stays close to the 'true' equilibrium during noise but adapts when there is a real trend. this is a fundamental tension that no single estimator resolves perfectly.

In [ ]:
# ----------------------------------------------------------------
# regime analysis for tomatoes
# ----------------------------------------------------------------

TOM_VOL_CALM_THRESH   = 1.5
TOM_VOL_ACTIVE_THRESH = 2.5

data_df = tom.copy()

# compute rolling volatility (same definition as trader_v6)
returns = data_df['mid_price'].diff()
rolling_vol = returns.rolling(20).std()
data_df['rolling_vol'] = rolling_vol

# assign regime label
def assign_regime(vol):
    if pd.isna(vol):
        return 'warmup'
    if vol < TOM_VOL_CALM_THRESH:
        return 'calm'
    if vol < TOM_VOL_ACTIVE_THRESH:
        return 'active'
    return 'volatile'

data_df['regime'] = data_df['rolling_vol'].apply(assign_regime)

print("regime distribution:")
print(data_df['regime'].value_counts())
print()

# compute mae per estimator per regime
regime_rows = []
for regime in ['calm', 'active', 'volatile']:
    regime_sub = data_df[data_df['regime'] == regime]
    if len(regime_sub) == 0:
        continue
    for name, col in ESTIMATORS.items():
        if col not in regime_sub.columns:
            continue
        errors = (regime_sub['mid_price'] - regime_sub[col]).dropna()
        mae = errors.abs().mean()
        regime_rows.append({
            'regime':    regime,
            'estimator': name,
            'mae':       round(mae, 4),
            'n':         len(errors),
        })

regime_df = pd.DataFrame(regime_rows)
# pivot for readability
regime_pivot = regime_df.pivot(index='estimator', columns='regime', values='mae')
print("mean absolute error (mae) of fv error by regime:")
print(regime_pivot.to_string())
print()
print("lower mae in calm = estimator stays close to price when market is quiet (good for spread detection)")
print("higher mae in volatile = estimator appropriately resists noise spikes (good for inventory protection)")

In [ ]:
# ----------------------------------------------------------------
# plot: mae by regime
# ----------------------------------------------------------------

regimes = ['calm', 'active', 'volatile']
estimator_list = list(ESTIMATORS.keys())
x = np.arange(len(regimes))
width = 0.12

fig, ax = plt.subplots(figsize=(12, 5))

for i, name in enumerate(estimator_list):
    if name not in regime_pivot.index:
        continue
    values = [regime_pivot.loc[name, r] if r in regime_pivot.columns else np.nan for r in regimes]
    offset = (i - len(estimator_list)/2) * width
    ax.bar(x + offset, values, width=width, label=name, color=colors_map.get(name, 'gray'), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.set_xlabel('volatility regime')
ax.set_ylabel('mean absolute error (price units)')
ax.set_title('tomatoes: fv estimator mae by volatility regime\n'
             'best for calm: small mae in calm. best for volatile protection: large mae in volatile.')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('regime_mae.png', bbox_inches='tight')
plt.show()

## section 9: spread size detection

a key use of the fv estimate in trader_v6 is to determine whether the current spread is wide enough to justify quoting. if the spread is narrow relative to the uncertainty in the fv estimate, quoting into it risks adverse selection.

here we compute a simple signal: the ratio of the observed spread to the rolling mae of the fv error.

spread_signal = observed_spread / rolling_mae_of_fv_error

a ratio much greater than 1 means the spread is wide relative to fv uncertainty (safe to quote). a ratio near 1 or below means the spread is thin relative to fv noise (risky to quote).

a good fv estimator for spread detection will produce a stable, low rolling mae so that the signal is precise and actionable.

In [ ]:
# ----------------------------------------------------------------
# spread detection capability
# ----------------------------------------------------------------

MAE_WINDOW = 50  # rolling window for mae of fv error

spread_rows = []
for name, col in ESTIMATORS.items():
    if col not in tom.columns:
        continue

    errors = (tom['mid_price'] - tom[col]).abs()
    rolling_mae = errors.rolling(MAE_WINDOW).mean()
    spread_signal = tom['spread'] / rolling_mae.replace(0, np.nan)

    spread_rows.append({
        'estimator':             name,
        'rolling_mae_mean':      round(rolling_mae.mean(), 4),
        'rolling_mae_std':       round(rolling_mae.std(), 4),
        'spread_signal_mean':    round(spread_signal.mean(), 3),
        'spread_signal_median':  round(spread_signal.median(), 3),
    })

spread_df = pd.DataFrame(spread_rows)
print("spread detection metrics (tomatoes):")
print(spread_df.to_string(index=False))
print()
print("rolling_mae_mean:      lower = estimator tracks price more tightly")
print("rolling_mae_std:       lower = more consistent (less surprise risk)")
print("spread_signal_mean:    higher = spread is more clearly wide relative to fv noise")
print("spread_signal_median:  same, using median to remove outlier influence")

## section 10: summary scorecard and recommendations

we synthesise the results across all three metrics into a single scorecard.

**scoring approach:**
each estimator is ranked 1-7 (1=best, 7=worst) on each metric dimension. the final score is the average rank across all dimensions.

**note on book-derived estimators:**
micro price and wall-mid score very well on half-life and mae metrics almost by construction: they are derived from the same book that generates the mid price, so their errors are trivially small. this does not mean they are the best fv estimators for inventory management or timing -- they have no smoothing, no memory, and no noise filtering. they are best interpreted as the 'raw signal' rather than the 'best estimate of equilibrium'.

**what we are confident about:**
1. emeralds: the constant fv=10000 is well-supported by the data. the mid price rarely deviates from 10000 and reverts within a handful of ticks when it does. there is no improvement available from any adaptive estimator here.
2. tomatoes: the ema-7 is a reasonable baseline but is not the best on any single metric. the kalman filter consistently matches or beats it on half-life while being more interpretable (its uncertainty estimate could be fed into the skew mechanism). the ou model is theoretically ideal but unreliable in practice with short windows.

**what we are uncertain about:**
- all estimates are in-sample. out-of-sample validation on a new day of data would sharply change the rankings for the more complex models.
- the ou and hybrid estimators are sensitive to their hyperparameters (window size, alpha). the numbers here reflect a single parameter choice.
- the 5-tick directional accuracy horizon is arbitrary. the rankings change at different horizons.

**recommendations by use case:**
- **regime detection:** kalman filter. its implicit variance estimate naturally distinguishes calm vs volatile periods.
- **inventory management and skewing:** kalman or ema-7. both are stable. ema-7 is simpler to reason about in live trading.
- **timing entries and exits:** micro price combined with ema-7 as a second check. micro price has the best short-horizon accuracy.
- **spread size detection:** ema-7 or kalman. their rolling mae is stable, making the spread/mae signal reliable.
- **overall:** the current ema-7 for tomatoes is good but not dominant. the kalman filter is the single best replacement if you want a single estimator for all tasks.

In [ ]:
# ----------------------------------------------------------------
# final scorecard: rank each estimator on each metric
# ----------------------------------------------------------------

# we build a score table for tomatoes (the product with non-trivial dynamics)
# emeralds is discussed separately since the constant is clearly dominant there

score_data = {}

# metric 1: half-life (lower = better)
hl_tom = hl_df[hl_df['product'] == 'TOMATOES'].copy()
hl_tom['hl_num'] = pd.to_numeric(hl_tom['half_life_ticks'], errors='coerce').fillna(1e6)
hl_tom['rank_hl'] = hl_tom['hl_num'].rank()
for _, row in hl_tom.iterrows():
    score_data.setdefault(row['estimator'], {})['rank_halflife'] = row['rank_hl']

# metric 2: z-score normality -- use abs(skewness) + abs(excess_kurtosis) as deviation score
# lower deviation = better
norm_tom = norm_df[norm_df['product'] == 'TOMATOES'].copy()
norm_tom['normality_dev'] = norm_tom['skewness'].abs() + norm_tom['ex_kurtosis'].abs()
norm_tom['rank_norm'] = norm_tom['normality_dev'].rank()
for _, row in norm_tom.iterrows():
    score_data.setdefault(row['estimator'], {})['rank_normality'] = row['rank_norm']

# metric 3: directional accuracy at h=5 (higher = better, so we rank ascending on 1-acc)
acc_tom_h5 = acc_df[(acc_df['product'] == 'TOMATOES') & (acc_df['horizon'] == 5)].copy()
acc_tom_h5['rank_acc'] = (1 - acc_tom_h5['acc_combined']).rank()
for _, row in acc_tom_h5.iterrows():
    score_data.setdefault(row['estimator'], {})['rank_accuracy'] = row['rank_acc']

# metric 4: spread detection -- rolling mae (lower = better)
spread_df['rank_mae'] = spread_df['rolling_mae_mean'].rank()
for _, row in spread_df.iterrows():
    score_data.setdefault(row['estimator'], {})['rank_spread'] = row['rank_mae']

# build final table
scorecard_rows = []
for est, scores in score_data.items():
    all_ranks = [v for v in scores.values() if not np.isnan(v)]
    avg_rank = np.mean(all_ranks) if all_ranks else np.nan
    scorecard_rows.append({
        'estimator':    est,
        'rank_halflife': scores.get('rank_halflife', np.nan),
        'rank_normality': scores.get('rank_normality', np.nan),
        'rank_accuracy': scores.get('rank_accuracy', np.nan),
        'rank_spread':  scores.get('rank_spread', np.nan),
        'avg_rank':     round(avg_rank, 2),
    })

sc_df = pd.DataFrame(scorecard_rows).sort_values('avg_rank')
print("final scorecard for tomatoes (lower avg_rank = better overall):")
print(sc_df.to_string(index=False))
print()
print("note: ranks are 1=best, 7=worst. the current ema-7 is highlighted below.")
print()
print(f"current ema-7 avg rank: {sc_df[sc_df['estimator']=='ema-7']['avg_rank'].values}")
print(f"best alternative:       {sc_df.iloc[0]['estimator']} (avg rank {sc_df.iloc[0]['avg_rank']})")

In [ ]:
# ----------------------------------------------------------------
# plot: scorecard heatmap
# ----------------------------------------------------------------

sc_plot = sc_df.set_index('estimator')[['rank_halflife','rank_normality','rank_accuracy','rank_spread']]

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(sc_plot.values.astype(float), cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=7)

ax.set_xticks(range(len(sc_plot.columns)))
ax.set_xticklabels(['half-life', 'normality', 'accuracy\n(h=5)', 'spread\ndetection'], fontsize=10)
ax.set_yticks(range(len(sc_plot.index)))
ax.set_yticklabels(sc_plot.index, fontsize=10)

# annotate each cell with the rank
for i in range(len(sc_plot.index)):
    for j in range(len(sc_plot.columns)):
        val = sc_plot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{int(round(val))}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if val > 4 else 'black')

plt.colorbar(im, ax=ax, label='rank (1=best, 7=worst)')
ax.set_title('tomatoes: estimator scorecard heatmap\ngreen = better rank, red = worse rank', fontsize=11)
plt.tight_layout()
plt.savefig('scorecard_heatmap.png', bbox_inches='tight')
plt.show()

## section 11: known limitations and caveats

this section is important. the numbers in this notebook should inform decisions, not dictate them. here are the key limitations you should keep in mind:

**1. in-sample evaluation**
all metrics are computed on the same data used to parameterise the estimators (e.g. the ema span of 7 was chosen from this dataset). the kalman filter parameters (q=0.5, r=1.0) and ou window (50 ticks) were similarly set by hand without a held-out validation set. switching to any of these alternative estimators based purely on in-sample results risks overfitting.

**2. data volume**
we have 20,000 ticks per product across 2 days. this is enough for broad comparisons but not for robust statistical testing of rare events. the directional accuracy estimates, particularly at horizon h=10, are based on fewer independent observations than they appear (consecutive ticks are correlated).

**3. book-derived estimators are not independent of mid price**
micro price and wall-mid are computed from the same order book that produces the mid price. their low mae and short half-life are partly circular: they trivially agree with the mid price because they are derived from the same best bid and ask that define the mid price. a more meaningful test would compare these estimators against the *next* mid price (1-tick-ahead prediction), not the current one.

**4. the ou and hybrid estimators have more tunable parameters**
the ou window (50 ticks), the hybrid alpha (0.5), and the ema spans (7/20) were chosen without optimisation. their relative performance would change with different parameter choices. this is a significant source of uncertainty when comparing them to the simpler ema-7.

**5. the constant emerald fv is robust but fragile**
the emerald mid price is almost perfectly pinned at 10000 in this dataset. however, this could change in future rounds if the product mechanics change. a very light kalman filter (q=0.01, r=1.0) would be a good safety net without sacrificing the near-zero tracking error seen in calm conditions.

**6. no execution or slippage model**
this analysis evaluates fv estimators on mid price. in live trading, orders execute at the bid or ask, not the mid. an estimator that generates better mid-price predictions does not necessarily generate better executions once the spread and queue position are accounted for.

In [ ]:
# ----------------------------------------------------------------
# final printed summary
# ----------------------------------------------------------------

print("=" * 65)
print("fair value estimator comparison: final summary")
print("=" * 65)
print()
print("--- emeralds ---")
print("the constant fv=10000 is well-supported by the data.")
print("mid price deviation from 10000 is extremely rare (std < 0.7).")
print("no alternative estimator offers a meaningful improvement.")
print("recommendation: keep constant fv=10000 for emeralds.")
print()
print("--- tomatoes ---")
print(sc_df[['estimator','avg_rank']].to_string(index=False))
print()
print("the current ema-7 is a solid baseline.")
print("the kalman filter is the best single replacement across all metrics.")
print("micro price is best for short-horizon accuracy but is too noisy for inventory management.")
print("the ou and hybrid estimators do not consistently outperform ema-7 on this dataset.")
print()
print("confident conclusions:")
print("  1. ema-7 is adequate. switching to kalman is a marginal improvement, not a step change.")
print("  2. using micro price as a secondary signal for entry timing has merit.")
print("  3. ou and hybrid are not recommended without further hyperparameter tuning and out-of-sample testing.")
print()
print("uncertain conclusions:")
print("  1. whether the kalman advantage holds out-of-sample (2 days of data is insufficient to say).")
print("  2. whether any fv estimator meaningfully improves pnl vs the execution and spread mechanisms.")
print("=" * 65)